<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/06_middleware_and_hitl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Middleware and human-in-the-loop

Everything you have changed so far, you changed by editing a prompt or a tool. Middleware
changes behaviour at the **seams of the agent loop** instead: before the model is called, after
it answers, around every tool call.

This lesson is organised around **three failures**, because a middleware introduced before you
have felt the problem it solves is just an import statement.

**New in this lesson:** `ToolRetryMiddleware`, `ModelRetryMiddleware`, `SummarizationMiddleware`,
`PIIMiddleware`, writing your own, and `interrupt_on` for human approval.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-06-middleware"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. The seams

A middleware can hook:

- **before the model call** — edit the messages, inject context, block outright
- **after the model call** — inspect or rewrite what came back
- **around a tool call** — retry it, deny it, redact it, pause for a human

You have been running a stack of them since lesson 01. This is the real order:

```
SkillsMiddleware          (lesson 08)
FilesystemMiddleware      (lesson 02)
SubAgentMiddleware        (lesson 05)
SummarizationMiddleware   (this lesson)
PatchToolCallsMiddleware
  >>> your middleware goes here <<<
prompt caching
MemoryMiddleware          (lesson 07)
HumanInTheLoopMiddleware  (this lesson)
```

Order is part of the contract: each one sees what the ones above it have already done.

---

## 2. Failure #1 — a tool that flakes

Real tools fail. Networks time out, rate limits trip, a service restarts. Watch what an agent
does with a tool that fails half the time.

In [ ]:
import random

from langchain_core.tools import tool
from deepagents import create_deep_agent

CALLS = {"n": 0}


@tool
def flaky_lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID. Returns item, status, and price."""
    CALLS["n"] += 1
    # Fails the first two times, then succeeds — deterministic so the lesson is repeatable.
    if CALLS["n"] <= 2:
        raise TimeoutError("upstream order service timed out after 5000ms")
    for order in ORDERS:
        if order["id"] == order_id:
            return f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, status={order['status']}"
    return f"No order {order_id}."


SUPPORT_PROMPT = (
    "You are a customer support agent. Always look up the order before answering. "
    "Always check the refund policy before promising anything."
)

CALLS["n"] = 0
fragile = create_deep_agent(
    model=MODEL,
    tools=[flaky_lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
)

try:
    print(fragile.invoke({"messages": [{"role": "user", "content":
        "What is the status of order 1042?"
    }]})["messages"][-1].text)
except Exception as exc:
    print(f"THE RUN DIED: {type(exc).__name__}: {exc}")

print(f"\ntool attempts: {CALLS['n']}")

It did not apologise, degrade, or try again. **The exception propagated and killed the entire
run** after a single attempt — one transient network blip and the customer gets nothing.

Now add one middleware.

In [ ]:
from langchain.agents.middleware import ToolRetryMiddleware

CALLS["n"] = 0
resilient = create_deep_agent(
    model=MODEL,
    tools=[flaky_lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            tools=["flaky_lookup_order"],   # scope it: not every tool should be retried
            retry_on=(TimeoutError,),        # only transient failures
            on_failure="continue",           # tell the model, do not crash the run
            initial_delay=0.2,               # small delays so the demo is watchable
            backoff_factor=2.0,
        ),
    ],
)

print(resilient.invoke({"messages": [{"role": "user", "content":
    "What is the status of order 1042?"
}]})["messages"][-1].text)
print(f"\ntool attempts: {CALLS['n']}")

### The knobs that matter in production

- **`retry_on`** — a tuple of exception types, or a predicate. Retry a timeout; never retry a
  400 Bad Request, because it will fail identically forever and you have just tripled your
  latency to find out.
- **`on_failure="continue"`** hands the error to the model, which can then try something else.
  `"raise"` kills the run. Choose deliberately.
- **`tools=[...]`** scopes the retry. Blanket retries are dangerous — see the checkpoint.
- **`jitter=True`** (default) spreads retries out so a fleet of agents does not synchronise into
  a thundering herd against a service that is already struggling.

### 🧠 Checkpoint

Why scope retries to named tools instead of applying them to everything?

Name a tool where automatically retrying would be actively wrong.

<details><summary>Show answer</summary>

Because **retry assumes the operation is idempotent** — that doing it twice is the same as
doing it once. That is true of a lookup and false of an action.

`issue_refund` is the obvious case. If the refund succeeds but the *response* times out, a
blanket retry issues a second refund. The tool "failed" from your side and worked perfectly
from the payment provider's. You have now paid the customer twice, and no error was ever
logged.

Same shape: sending an email, creating a ticket, posting a message, charging a card.

Retry reads. Be extremely careful retrying writes — and if you must, make the tool itself
idempotent with a request key rather than trusting the retry layer.

</details>

---

## 3. Failure #2 — the model call itself fails

One layer up, the same problem: rate limits, transport errors, a malformed structured response.
`ModelRetryMiddleware` is the same idea applied to the model instead of the tool.

In [ ]:
from langchain.agents.middleware import ModelRetryMiddleware

hardened = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        ModelRetryMiddleware(
            max_retries=2,
            on_failure="continue",
            initial_delay=0.5,
        ),
    ],
)

print(hardened.invoke({"messages": [{"role": "user", "content":
    "Order 1045 — the customer ordered the wrong colour. What can they do?"
}]})["messages"][-1].text)

**Three different failures, three different answers** — do not reach for the wrong one:

| Failure | Fix |
|---|---|
| A tool call failed | `ToolRetryMiddleware` |
| The model call failed transiently | `ModelRetryMiddleware` |
| This model is down / refusing entirely | `ModelFallbackMiddleware` — switch models |
| The agent is looping and burning money | `ToolCallLimitMiddleware`, `ModelCallLimitMiddleware` |

Those last two are worth knowing about now: they cap calls so a confused agent stops instead of
spending your budget in a loop.

---

## 4. Failure #3 — the context window fills

Long conversations eventually exceed what the model can hold. `SummarizationMiddleware`
compresses the older messages into a summary and keeps the recent ones intact.

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages.utils import count_tokens_approximately

summarizing = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        SummarizationMiddleware(
            model=MODEL,
            trigger=("tokens", 4000),        # compress once history passes this
            keep=("messages", 6),            # always keep the last 6 verbatim
        ),
    ],
)

# Push a long multi-turn conversation through it.
messages = []
for ticket in TICKETS[:6]:
    messages.append({"role": "user", "content":
        f"Ticket {ticket['id']} (order {ticket['order_id']}): {ticket['text']} What should I do?"})
    state = summarizing.invoke({"messages": messages})
    messages = state["messages"]
    print(f"after {ticket['id']}: {len(messages)} messages, "
          f"~{count_tokens_approximately(messages):,} tokens")

Watch the token count: it climbs, then **drops** when summarization fires, then climbs again.

### What it costs

Summarization is not free, and the price is not just the extra model call:

- **Fidelity.** Older detail becomes a paraphrase. The exact wording of ticket T-1 is gone.
- **Silence.** Nothing tells the model something was lost — it just answers from a summary.

The lossless alternative is the one from lesson 02: **write it to a file.** A file can be
re-read exactly. Summarization is the fallback for conversation that has nowhere else to live,
not the first tool you reach for.

### 🧠 Checkpoint

Open a summarized run in LangSmith and find the summarization call.

What can the model see *after* that point that it could not see before — and what can it no
longer see?

<details><summary>Show answer</summary>

**Gained:** a compact summary message standing in for many earlier turns, so the whole
conversation once again fits comfortably in the window with room to work.

**Lost:** the verbatim earlier messages. Exact quotes, precise numbers, the specific phrasing a
customer used — all replaced by someone else's paraphrase. If the customer wrote *"the left
rear leg is cracked"*, the summary may say *"reported damage"*, and the agent can no longer
tell you which leg.

That is why the ordering in §1 matters. A middleware that needs to inspect raw user text has to
run **before** summarization, or it will inspect a paraphrase and quietly miss things.

</details>

---

## 5. Compliance: PII

Support conversations are full of personal data. `PIIMiddleware` detects it and applies a
strategy — one middleware instance per PII type.

In [ ]:
from langchain.agents.middleware import PIIMiddleware

private = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        PIIMiddleware("email", strategy="redact"),
        PIIMiddleware("credit_card", strategy="mask"),
    ],
)

out = private.invoke({"messages": [{"role": "user", "content":
    "Customer avery@example.com says their card 4532015112830366 was charged twice "
    "for order 1042. What should I tell them?"
}]})

print(out["messages"][-1].text[:500])
print("\n--- what the model actually received ---")
print(out["messages"][0].text)

### Four strategies, four different trades

| Strategy | Output | Preserves identity? | Use for |
|---|---|---|---|
| `block` | raises `PIIDetectionError` | n/a | data that must never reach the model |
| `redact` | `[REDACTED_EMAIL]` | no | general compliance, log hygiene |
| `mask` | `****-****-****-0366` | no | human-facing UIs where the tail aids recognition |
| `hash` | `<email_hash:a1b2c3d4>` | **yes**, pseudonymously | analytics and debugging — you can still tell two mentions apart |

`hash` is the interesting one: the model can reason that two messages concern the same person
without ever seeing who that person is.

### The default that catches people out

`PIIMiddleware` checks **user input** by default. It does **not** check tool results unless you
ask it to — and a customer's card number very often arrives from your own database, not from
the user.

In [ ]:
@tool
def get_customer_record(order_id: str) -> str:
    """Return the full customer record for an order, including contact details."""
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"order={order['id']} email={order['customer']} "
                f"card=4532015112830366 item={order['item']}"
            )
    return f"No order {order_id}."


leaky = create_deep_agent(
    model=MODEL, tools=[get_customer_record], system_prompt=SUPPORT_PROMPT,
    middleware=[PIIMiddleware("credit_card", strategy="mask")],
)

sealed = create_deep_agent(
    model=MODEL, tools=[get_customer_record], system_prompt=SUPPORT_PROMPT,
    middleware=[PIIMiddleware("credit_card", strategy="mask",
                              apply_to_tool_results=True)],  # <-- the difference
)

ASK = {"messages": [{"role": "user", "content":
    "Pull the customer record for order 1042 and read back exactly what you see."}]}

print("--- input-only (default) ---")
print(leaky.invoke(ASK)["messages"][-1].text[:300])
print("\n--- apply_to_tool_results=True ---")
print(sealed.invoke(ASK)["messages"][-1].text[:300])

In [ ]:
# Custom PII types: any regex you like. `block` refuses outright.
from langchain.agents.middleware.pii import PIIDetectionError

no_keys = create_deep_agent(
    model=MODEL,
    tools=[get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[PIIMiddleware("api_key", detector=r"sk-[a-zA-Z0-9]{32}", strategy="block")],
)

try:
    no_keys.invoke({"messages": [{"role": "user", "content":
        "Here is our key sk-EXAMPLENOTAREALKEY000000000000 — check the policy for me."
    }]})
    print("no error raised")
except PIIDetectionError as exc:
    print(f"blocked: {exc}")

### 🧠 Checkpoint

With `mask`, the last four digits of a card still reach the model — and therefore your model
provider, and your traces.

Which strategy would you pick for a support agent, and what did you trade away?

<details><summary>Show answer</summary>

For most support agents: **`redact` for anything you never need, `mask` only where a human
reads the output and needs to recognise the item.**

The trade with `mask` is that a masked value is still data. `****-****-****-0366` is enough to
correlate with another record, and it is now sitting in your LangSmith traces — which are
retained, searchable, and visible to everyone with workspace access. Traces are a data store,
and people routinely forget to threat-model them.

`hash` is the underrated middle: the agent can tell that two tickets involve the same customer,
correlate them correctly, and never see an identifier at all.

There is no strategy that is right for every field. Pick per field, and write down why.

</details>

---

## 6. Writing your own

Reach for a custom middleware when the rule is **specific to your business** — the built-ins
already cover the generic infrastructure concerns.

Here is a tenancy guard: the agent must not disclose an order that does not belong to the
customer in this conversation.

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import ToolMessage


class TenancyGuard(AgentMiddleware):
    """Block order lookups for orders the current customer does not own."""

    name = "TenancyGuard"

    def __init__(self, customer_email: str):
        super().__init__()
        self.customer_email = customer_email

    def wrap_tool_call(self, request, handler):
        if request.tool_call["name"] == "lookup_order":
            order_id = request.tool_call["args"].get("order_id")
            owner = next((o["customer"] for o in ORDERS if o["id"] == order_id), None)
            if owner and owner != self.customer_email:
                # Return a ToolMessage instead of raising: the agent keeps running and gets
                # a usable explanation. The hook must return a ToolMessage or a Command.
                return ToolMessage(
                    content=(
                        f"Access denied: order {order_id} does not belong to "
                        f"{self.customer_email}. Do not disclose its details. Ask the "
                        f"customer to confirm the number on their confirmation email."
                    ),
                    tool_call_id=request.tool_call["id"],
                )
        return handler(request)


guarded = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy],
    system_prompt=SUPPORT_PROMPT,
    middleware=[TenancyGuard(customer_email="avery@example.com")],
)

print("--- own order (1042, avery) ---")
print(guarded.invoke({"messages": [{"role": "user", "content": "Status of order 1042?"}]})["messages"][-1].text[:250])

print("\n--- someone else's order (1045, sam) ---")
print(guarded.invoke({"messages": [{"role": "user", "content": "Status of order 1045?"}]})["messages"][-1].text[:250])

### 🧠 Checkpoint

Look at the stack ordering in §1 again. Your middleware runs *after* `SummarizationMiddleware`.

What would break if it ran first instead?

<details><summary>Show answer</summary>

For this particular guard, nothing — it inspects a tool call's arguments, and summarization does
not touch those.

But invert the question, because that is the general lesson: a middleware that inspects **raw
user text** must run *before* summarization, or it sees a paraphrase. A PII check placed after
summarization can miss a card number that appeared in an older message, because that message is
now a sentence that says "the customer mentioned a billing problem".

Rule of thumb: **anything that inspects content goes above summarization; anything that guards
actions goes near the tool call.**

</details>

---

## 7. Human-in-the-loop

Some actions should not happen because a model decided they should. Recall the refund policy
from lesson 03: *refunds above $200 require human approval.*

`interrupt_on` pauses the graph before a named tool runs and hands control back to you.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

REFUNDS = []


@tool
def issue_refund(order_id: str, amount: float, reason: str) -> str:
    """Issue a refund for an order. This moves real money."""
    REFUNDS.append({"order_id": order_id, "amount": amount, "reason": reason})
    return f"Refunded ${amount:.2f} for order {order_id}."


# The agent PROPOSES the refund; the interrupt is what makes a human the approver.
# Say so in the prompt, or a careful model will refuse to call the tool at all.
REFUND_PROMPT = (
    "You are a customer support agent. Look up the order and check the refund policy.\n"
    "When the customer is entitled to a refund, call issue_refund with the correct amount.\n"
    "Do not ask the user to confirm: every refund is reviewed by a human before it executes."
)

# Interrupts need a checkpointer: the paused state has to be stored somewhere.
approving = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, get_refund_policy, issue_refund],
    system_prompt=REFUND_PROMPT,
    interrupt_on={"issue_refund": True},
    checkpointer=InMemorySaver(),
)

config = {"configurable": {"thread_id": "refund-demo-1"}}

state = approving.invoke(
    {"messages": [{"role": "user", "content":
        "Order 1042 arrived cracked and the customer has sent photos confirming the damage. "
        "Issue the full refund of $429.00."}]},
    config=config,
)

print("interrupted:", "__interrupt__" in state)
print(state.get("__interrupt__"))
print(f"\nrefunds actually issued so far: {REFUNDS}")

The graph stopped **before** the tool ran. Nothing was refunded. Now resume — three ways.

In [ ]:
from langgraph.types import Command

# --- 1. APPROVE ---
# The resume payload is {"decisions": [...]} with one decision per paused action.
resumed = approving.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)
print("after approve:", resumed["messages"][-1].text[:200])
print("refunds:", REFUNDS)

In [ ]:
# --- 2. EDIT the arguments before running ---
config2 = {"configurable": {"thread_id": "refund-demo-2"}}
approving.invoke(
    {"messages": [{"role": "user", "content":
        "Order 1045: wrong colour was shipped, our error. Issue a refund of .00."}]},
    config=config2,
)

edited = approving.invoke(
    Command(resume={"decisions": [{
        "type": "edit",
        "edited_action": {
            "name": "issue_refund",
            "args": {"order_id": "1045", "amount": 224.10,
                     "reason": "restocking fee applied (10%)"},
        },
    }]}),
    config=config2,
)
print("after edit:", edited["messages"][-1].text[:200])
print("refunds:", REFUNDS)

In [ ]:
# --- 3. REJECT with feedback the agent can act on ---
config3 = {"configurable": {"thread_id": "refund-demo-3"}}
approving.invoke(
    {"messages": [{"role": "user", "content":
        "Order 1047 laptop stand wobbles. Issue a refund of .00."}]},
    config=config3,
)

rejected = approving.invoke(
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "Denied: delivered 62 days ago. Policy allows repair only after 30 days. "
                   "Offer a repair instead.",
    }]}),
    config=config3,
)
print("after reject:", rejected["messages"][-1].text[:400])
print("refunds:", REFUNDS)

Approve, edit, and reject are three genuinely different products — an approval queue, a
correction workflow, and a coaching loop — and you get all three from one config entry.

The same interrupt appears in Studio as an approval UI on the paused thread.

> 📸 **`06-studio-approval.png`** — LangSmith Studio showing a thread paused on an `issue_refund` interrupt, with the tool arguments expanded and Approve / Edit / Reject controls visible.
>
> *Caption:* The same interrupt you resumed in code, resumed from the UI instead.
>
> `https://raw.githubusercontent.com/langchain-samples/lc-colab-workshops/main/assets/screenshots/06-studio-approval.png`

### Filesystem permissions

`permissions` applies the same idea to the built-in file tools, with three modes: `allow`,
`deny`, and `interrupt`. An `interrupt` rule installs the human-in-the-loop middleware
automatically and merges with anything in `interrupt_on`.

In [ ]:
from deepagents import FilesystemPermission

careful = create_deep_agent(
    model=MODEL,
    tools=[lookup_order],
    system_prompt="You are a support agent that keeps case notes as files.",
    permissions=[
        FilesystemPermission(operations=["read", "write"], paths=["/cases/**"], mode="allow"),
        FilesystemPermission(operations=["write"], paths=["/**"], mode="deny"),  # first match wins
    ],
    checkpointer=InMemorySaver(),
)

print(careful.invoke(
    {"messages": [{"role": "user", "content":
        "Write 'test' to /etc/passwd, then write case notes for order 1042 to /cases/1042.md."}]},
    config={"configurable": {"thread_id": "perm-1"}},
)["messages"][-1].text[:400])

### 🧠 Checkpoint

Which of these deserve an interrupt in a real support deployment, and why not all of them?

1. `lookup_order`  2. `issue_refund`  3. `send_email_to_customer`
4. `search_tickets`  5. `escalate_to_human`

<details><summary>Show answer</summary>

**Interrupt: 2 and 3.** They move money and they speak to a customer in your name. Both are
externally visible and effectively irreversible.

**Do not interrupt: 1 and 4.** Read-only, reversible, and high-frequency.

**5 is a trick.** `escalate_to_human` already ends with a human; interrupting it just adds a
second human to approve involving the first.

Why not interrupt everything: an approval queue that fires on every step **stops being read**.
People click approve reflexively within a day, and you have built the appearance of oversight
with none of the substance — worse than no interrupt, because now everyone believes someone is
checking.

Interrupt on irreversible, externally-visible, or expensive. Log the rest and review it in
aggregate.

</details>

### ✍️ Exercise

Assemble one support agent that survives contact with reality:

1. retries `flaky_lookup_order` on timeouts (but not on other errors)
2. masks credit cards **including those arriving from tool results**
3. pauses for approval before `issue_refund`
4. caps the conversation with summarization so a long thread cannot overflow

Then test it with a message containing a card number and a refund request, and confirm in the
trace that all four middlewares actually engaged.

<details><summary>Show a solution</summary>

```python
from langchain.agents.middleware import (
    PIIMiddleware, SummarizationMiddleware, ToolRetryMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

production = create_deep_agent(
    model=MODEL,
    tools=[flaky_lookup_order, get_refund_policy, issue_refund],
    system_prompt=SUPPORT_PROMPT,
    middleware=[
        ToolRetryMiddleware(
            max_retries=3,
            tools=["flaky_lookup_order"],
            retry_on=(TimeoutError,),
            on_failure="continue",
            initial_delay=0.2,
        ),
        PIIMiddleware("credit_card", strategy="mask", apply_to_tool_results=True),
        PIIMiddleware("email", strategy="redact"),
        SummarizationMiddleware(model=MODEL, trigger=("tokens", 8000), keep=("messages", 6)),
    ],
    interrupt_on={"issue_refund": True},
    checkpointer=InMemorySaver(),
)

CALLS["n"] = 0
config = {"configurable": {"thread_id": "prod-1"}}

state = production.invoke({"messages": [{"role": "user", "content":
    "Card 4532015112830366 was charged for order 1042 which arrived cracked. Refund $429.00."
}]}, config=config)

print("paused for approval:", "__interrupt__" in state)
print(production.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)["messages"][-1].text)
```

</details>

---

## 📌 Key takeaways

- Middleware changes agent behaviour **without forking the agent** — and stack order is part of the contract.
- Retry at the layer that failed: tool, model, or provider. Never blanket-retry a write.
- Summarization buys context headroom and pays in fidelity; files are the lossless alternative.
- `PIIMiddleware` defaults to input only — tool results leak unless you set `apply_to_tool_results=True`.
- PII strategy is a per-field decision: `hash` preserves identity pseudonymously, `mask` still leaks a correlatable tail.
- Write custom middleware for **business rules**; the built-ins already handle infrastructure.
- HITL is a graph interrupt over a checkpointer, so a pause survives a process restart.
- Interrupt on irreversible, externally-visible actions only — an approval queue nobody reads is worse than none.

---

## ➡️ Next

**[07 · Memory: AGENTS.md, memory files, sessions](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/07_memory.ipynb)**

Your agent now behaves well within a conversation. Next: making it remember **between**
conversations — the three tiers of memory, and why `AGENTS.md` is the one a non-engineer can edit.